# 02 — Build an inspectable local RAG assistant

**Track:** Beginner · **Stage:** Local Implementation

By the end of this notebook, you will build, run, debug, and evaluate a dependency-free local RAG assistant using **LangChain**, **HuggingFace Embeddings**, and **ChromaDB**. You will explain every score it produces, build a bounded context window, and return citations.

> The goal is not to make a clever chatbot. It is to establish a transparent baseline where you can inspect the exact evidence retrieved before generating an answer.

## Setup: Local Models and Vector Database

We will use `HuggingFaceEmbeddings` for local, cost-free vectorization, and `Chroma` for our local vector store. For generation, we mock the LLM output to keep this notebook fully local and API-key free, but the architecture perfectly mirrors production setups.

In [ ]:
# !pip install langchain langchain-huggingface chromadb sentence-transformers

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 1. Ingestion: Document to Vector

In a real system, documents are parsed, chunked, and enriched with metadata. Here, we define a small corpus manually to maintain full visibility.

In [ ]:
corpus = [
    Document(
        page_content="Harborline Support Escalation Policy: Tier 1 may reboot edge nodes. Tier 2 must approve any database failover.",
        metadata={"source": "escalation_policy.md", "section": "permissions", "id": "chunk-01"}
    ),
    Document(
        page_content="Incident Response: If the payments API returns 503, immediately check the Stripe gateway status page before paging on-call.",
        metadata={"source": "incident_response.md", "section": "payments", "id": "chunk-02"}
    ),
    Document(
        page_content="System Architecture: The payments API relies on a PostgreSQL cluster in the us-east-1 region.",
        metadata={"source": "architecture.md", "section": "database", "id": "chunk-03"}
    )
]

# Initialize local embeddings (downloads a small model the first time)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create an ephemeral local vector store
vectorstore = Chroma.from_documents(corpus, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

## 2. Inspecting Retrieval (The Evidence Boundary)

Before we pass anything to an LLM, we must verify the retriever finds the correct evidence. Dense embeddings capture semantic meaning, not just exact keyword matches.

In [ ]:
question = "Who is allowed to trigger a DB failover?"

# Using similarity_search_with_score to inspect the raw distance metrics
results = vectorstore.similarity_search_with_score(question, k=2)

print(f"Question: {question}\n")
for doc, score in results:
    # Note: Chroma returns distance metrics (lower is closer/better)
    print(f"[Score: {score:.4f}] Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}\n")

## 3. Constructing the Chain

Now we bind the retriever to the generation step, injecting the document metadata so the LLM can cite its sources.

In [ ]:
def format_docs_with_citations(docs):
    return "\n\n".join(f"[Citation: {d.metadata['source']}] {d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_template(
    "Answer the user's question based strictly on the context below. "
    "Include the [Citation: ...] in your answer.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

llm = FakeListLLM(responses=["Based on [Citation: escalation_policy.md], Tier 2 must approve any database failover."])

chain = (
    {"context": retriever | format_docs_with_citations, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Final Answer:")
print(chain.invoke(question))

## 4. Debugging the Trace

If the answer is wrong, where did the pipeline fail? 
1. **Ingestion failure:** The document wasn't in the vector store.
2. **Retrieval failure:** The retriever scored irrelevant documents higher than the correct one.
3. **Generation failure:** The retriever found the correct context, but the LLM hallucinated or ignored it.

By exposing `similarity_search_with_score` before the LLM step, you can instantly determine if a failure is a retrieval issue (search problem) or a generation issue (prompt/model problem).